<a href="https://colab.research.google.com/github/JulTob/BasesDeDatos/blob/main/%F0%9D%99%B9%F0%9D%9A%83_BasesDatos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🄱🄰🅂🄴🅂 🄳🄴 🄳🄰🅃🄾🅂**

# 𝙹𝚞𝚕𝚒𝚘 𝚃𝚘𝚋𝚘𝚜𝚘
### **2025**

---

###  Esquema metodológico de trabajo

Se introducen ejemplos básicos de SQL, paralelos a los operadores del álgebra relacional.

Las uniones de las tablas se tienen que realizar bien mediante el comando **WHERE** o, si fuera necesario, mediante los comandos **LEFT JOIN** o **RIGHT JOIN**, y **no con el uso del comando INNER JOIN**.

---

### Enunciado

Descargar la Base de Datos **“Human Resources” (HR)**, revisar atentamente el esquema relacional y realizar las consultas SQL.

---




Se ha usado el entorno Python para preparar a sqlite3, en una llamada funcional que importa los datos del entorno GitHub, después de haberlos pasado a un formato csv que permite una carga rápida y eficiente.

Las tablas han recibido nombres para el entorno SQL, e internamente se han guardado como bases de datos (dataframes) de NumPy. Esto solo afecta a que podemos usar la función display para enseñar el Head+Tail de los datos obtenidos por la Query. La ventaja es que tablas largas se colapsan automáticamente.

In [55]:
import pandas as pd
import sqlite3

base_url = "https://raw.githubusercontent.com/JulTob/Data/main/PD/HR/"

table_files = {
    "COUNTRIES":   "HR_countries.csv",
    "DEPARTMENTS": "HR_departments.csv",
    "EMPLOYEES":   "HR_employees.csv",
    "JOB_HISTORY": "HR_job_history.csv",
    "JOBS":        "HR_jobs.csv",
    "LOCATIONS":   "HR_locations.csv",
    "REGIONS":     "HR_regions.csv",
}

dfs = {}
for table_name, filename in table_files.items():
    url = base_url + filename
    dfs[table_name] = pd.read_csv(url)

# Create a SQLite in-memory database (or specify a file path like 'HR.sqlite')
conn = sqlite3.connect(":memory:") # Using ':memory:' for an in-memory database

for table_name, df in dfs.items():
    df.to_sql(table_name, conn, if_exists="replace", index=False)

print("SQLite database created with tables:")
# Query to check table names
query_tables = "SELECT name FROM sqlite_master WHERE type='table';"
display(pd.read_sql_query(query_tables, conn))

# The connection 'conn' is now open and contains your data, ready for queries
# We will keep the connection open for subsequent queries.

SQLite database created with tables:


,name
0,COUNTRIES
1,DEPARTMENTS
2,EMPLOYEES
3,JOB_HISTORY
4,JOBS
5,LOCATIONS
6,REGIONS


Se ha realizado una función que aisla el procedimiento interno, permitiendo centrarnos en las solicitudes de SQL.

Además, la función aporta invariablemente el tamaño de los datos solicitados, con un print del número de entradas, los datos por entrada, y el total de datos recogidos.

In [56]:

def SQL(sql_query):
    df_result = pd.read_sql_query(sql_query, conn)
    display(df_result)
    rows, cols = df_result.shape
    print(f"{rows}x{cols}: {rows * cols} data elements")


sql_query = """
Select *
From COUNTRIES;
"""


SQL(sql_query)

,country_id,country_name,region_id
0,AR,Argentina,2
1,AU,Australia,3
2,BE,Belgium,1
3,BR,Brazil,2
4,CA,Canada,2
5,CH,Switzerland,1
6,CN,China,3
7,DE,Germany,1
8,DK,Denmark,1
9,EG,Egypt,4


25x3: 75 data elements


In [57]:
SQL("""
Select *
From DEPARTMENTS;
""")

,department_id,department_name,manager_id,location_id
0,10,Administration,200.0,1700
1,20,Marketing,201.0,1800
2,30,Purchasing,114.0,1700
3,40,Human Resources,203.0,2400
4,50,Shipping,121.0,1500
5,60,IT,103.0,1400
6,70,Public Relations,204.0,2700
7,80,Sales,145.0,2500
8,90,Executive,100.0,1700
9,100,Finance,108.0,1700


27x4: 108 data elements


En este diagrama Podemos consultar la estructura ER simplificada. El tenerla en texto permite consultar nombres y copiarlos de manera sencilla para las *queries*.

```text
Legend:
⬜️ = Primary Key (Color matches table)
⚪️ = Foreign Key (Color matches related table)

Table Color Codes:
🟥 = REGIONS
🟧 = COUNTRIES
🟨 = LOCATIONS
🟩 = DEPARTMENTS
🟦 = EMPLOYEES
🟪 = JOBS
🟫 = JOB_HISTORY


┌───────────────────┐
│ 🟥     REGIONS    │
├───────────────────┤
│ 🟥 REGION_ID      │
│    REGION_NAME    │
└───────────────────┘

┌─────────────────────┐
│ 🟧     COUNTRIES    │
├─────────────────────┤
│ 🟧 COUNTRY_ID       │
│    COUNTRY_NAME     │
│ 🔴 REGION_ID        │ ⟵ | (1:M) has | ⟵  🟥 REGIONS
└─────────────────────┘

┌───────────────────────┐
│ 🟨     LOCATIONS      │
├───────────────────────┤
│ 🟨 LOCATION_ID        │
│    STREET_ADDRESS     │
│    POSTAL_CODE        │
│    CITY               │
│    STATE_PROVINCE     │
│ 🟠 COUNTRY_ID         │ ⟵ | (1:M) has | ⟵ 🟧 COUNTRIES
└───────────────────────┘

┌─────────────────────────┐
│ 🟩     DEPARTMENTS      │
├─────────────────────────┤
│ 🟩 DEPARTMENT_ID        │
│    DEPARTMENT_NAME      │
│ 🔵 MANAGER_ID           │ ⟵ | (1:M) manages | ⟵ 🟦EMPLOYEES
│ 🟡 LOCATION_ID          │ ⟵ | (1:M) hosts | ⟵ 🟨LOCATIONS
└─────────────────────────┘

┌─────────────────────┐
│ 🟦     EMPLOYEES    │
├─────────────────────┤
│ 🟦 EMPLOYEE_ID      │
│    FIRST_NAME       │
│    LAST_NAME        │
│    EMAIL            │
│    PHONE_NUMBER     │
│    HIRE_DATE        │
│ 🟣 JOB_ID           │ ⟵ | (1:M) assigned to | ⟵ 🟪JOBS
│    SALARY           │
│    COMMISSION_PCT   │
│ 🔵 MANAGER_ID       │ ⟵ | (1:M) supervises | ⟵ 🟦EMPLOYEES
│ 🟢 DEPARTMENT_ID    │ ⟵ | (1:M) belongs to | ⟵ 🟩DEPARTMENTS
└─────────────────────┘

┌─────────────────────┐
│ 🟫   JOB_HISTORY    │
├─────────────────────┤
│ 🟫🔵 EMPLOYEE_ID     │ ⟵ | (1:M) has history | ⟵ 🟦EMPLOYEES
│ 🟫 START_DATE       │
│    END_DATE         │
│ 🟣 JOB_ID           │ ⟵ | (1:M) recorded in | ⟵ 🟪JOBS
│ 🟢 DEPARTMENT_ID    │ ⟵ | (1:M) includes | ⟵ 🟩DEPARTMENTS
└─────────────────────┘

┌────────────────┐
│ 🟪     JOBS    │
├────────────────┤
│ 🟪 JOB_ID      │
│    JOB_TITLE   │
│    MIN_SALARY  │
│    MAX_SALARY  │
└────────────────┘
```

## Ejercicio a)
Mostrar el Nombre, Apellidos y número de teléfono de los empleados cuyo trabajo sea de vendedor (es decir que job_id sea IT_PROG).

1. ## $R_1$ := 𝜎(JOB_ID = "IT_PROG")[EMPLOYEES]
2. ## $R_2$ := 𝜋(FIRST_NAME, LAST_NAME, PHONE_NUMBER)[$R_1$]

In [58]:
SQL("""
SELECT
    first_name as Nombre,
    last_name as Apellidos,
    phone_number as Telefono
FROM EMPLOYEES
WHERE JOB_ID = 'IT_PROG'
;
""")

,Nombre,Apellidos,Telefono
0,Alexander,Hunold,590.423.4567
1,Bruce,Ernst,590.423.4568
2,David,Austin,590.423.4569
3,Valli,Pataballa,590.423.4560
4,Diana,Lorentz,590.423.5567


5x3: 15 data elements


## Ejercicio b)
Dar una lista con el Nombre y Apellidos de Empleados que no tengan comisión asignada (COMMISSION_PCT).

1. 𝜎(COMMISSION_PCT is ␀)[EMPLOYEES]
2. 𝜋(FIRST_NAME, LAST_NAME)[1.]

In [59]:
SQL("""
SELECT
    first_name as Nombre,
    last_name as Apellidos
FROM EMPLOYEES
WHERE  COMMISSION_PCT IS 0
    OR COMMISSION_PCT IS NULL
;
""")

,Nombre,Apellidos
0,Steven,King
1,Neena,Kochhar
2,Lex,De Haan
3,Alexander,Hunold
4,Bruce,Ernst
...,...,...
102,Pat,Fay
103,Susan,Mavris
104,Hermann,Baer
105,Shelley,Higgins


107x2: 214 data elements


In [61]:
SQL("""
SELECT
    "Sin Comision" as Tipo,
    COUNT(*) as Total FROM EMPLOYEES
        WHERE COMMISSION_PCT IS NULL
UNION SELECT
    "Con Comision" as Tipo,
    COUNT(*) as Total FROM EMPLOYEES
    WHERE COMMISSION_PCT IS NOT NULL
;
""")

,Tipo,Total
0,Con Comision,35
1,Sin Comision,72


2x2: 4 data elements


## Ejercicio c)
Obtener una lista donde se indique cuantos empleados tiene comisión y cuantos no.

> Nota: El planteamiento tiene cierta ambiguedad. Por un lado en entorno técnico, entenderíamos que son los valores nulos en comisiones, _peeeero_ en entornos laborales, como en ofimática, esta pregunta solicitaría también aquellos que tienen un valor de 0 en el campo no-nulo. Es interesante investigar ambos planteamientos. Voy a tomar esta ruta dual.

1. E := 𝜌[EMPLOYEE]
2. A := 𝜎(COMISSION is ␀)[E]
2. B := 𝜎(COMISSION is 0)[E]
2. C := 𝜎(COMISSION > 0)[E]
3. ⏎ R := |A| ∪ |B| ∪ |C|

In [ ]:
SQL("""
SELECT
    COUNT(*) AS TOTAL,
    COUNT(COMMISSION_PCT) AS CON_COMISION,
    COUNT(*) - COUNT(COMMISSION_PCT) AS SIN_COMISION
FROM EMPLOYEES;
""")

Truco: Se puede hacer un **histograma** analítico de los datos con la función de COUNT aplicada a un GROUP-BY. Esto separa los tipos de la columna, con cada count aplicada a cada grupo en lugar de al total de tabla.

In [ ]:
SQL("""
SELECT
    COMMISSION_PCT as Comision,
    COUNT(COMMISSION_PCT) as Conteo
FROM EMPLOYEES
GROUP BY COMMISSION_PCT
;
""")

Esta solución la marco como _Truco_ porque en otros contextos o en valores continuos de Commission_PCT podría potencialmente crear una tabla enorme. Aunque también podríamos extraer la información de esa tabla con `where`.

Alternativamente, la forma más meticulosa sería así:

In [ ]:
SQL("""
SELECT
    COUNT(COMMISSION_PCT) as "Known",
    SUM(COMMISSION_PCT IS 0) as "Comission=0",
    SUM(COMMISSION_PCT IS NOT NULL
        AND COMMISSION_PCT IS NOT 0) as "Comission>0",
    SUM(COMMISSION_PCT IS NULL) as "Unknown",
    SUM(COMMISSION_PCT) as "Total Comissions"

FROM EMPLOYEES
;
""")

> Tenemos que usar SUM(condición), donde cada entrada evalua la condición en esa instancia. En caso afirmativo devuelve 1, con lo que se suma al tallaje total.

> PD: Sabemos que solo podemos tener comisiones >0 por lógica del negocio (Bussiness Logic).

## Ejercicio d)
Obtener una lista de empleados con el nombre, los apellidos, el email y el salario de cada uno éstos ordenada de menor a mayor salario.

$$
𝜗:=\text{FIRST_NAME},\ \text{LAST_NAME},\ \text{EMAIL},\ \text{SALARY}
$$
$$
R₁ :=
𝜋_{𝜗}
[\text{EMPLOYEES}]
$$

$$
𝜏_{\text{SALARY}↑}[R_1]
$$

In [ ]:
SQL("""
SELECT
    first_name AS Nombre,
    last_name AS Apellidos,
    email AS Email,
    salary AS Salario
FROM EMPLOYEES
ORDER BY SALARY
;
""")

## Ejercicio e)
Realizar una lista como la anterior añadiendo además la columna correspondiente al nombre del departamento. ¿Qué se observa respecto al número de filas obtenidas en esta consulta respecto a la anterior? ¿Qué explicación puedes dar al respecto?

$$
E:= 𝛒(EMPLOYEES)
$$
$$
D:= 𝛒(DEPARTMENTS)
$$

$$
a:= E.Department＿ID = D.Department＿ID
$$

$$
ED :=
E \underset{a}{⋈} D
$$

$$
b :=
{
FIRST＿NAME,
LAST＿NAME,
EMAIL,
SALARY,
DEPARTMENT＿NAME
}
$$

$$
R₁ := 𝜋_{(b)}[ED]
$$

$$
R :=
\tau_{\text{SALARY} \uparrow}(R_1)
$$


Tú decides el siguiente paso.


In [ ]:
SQL("""
SELECT
    E.first_name AS Nombre,
    E.last_name AS Apellidos,
    E.email AS Email,
    E.salary AS Salario,
    D.department_name AS Departamento
FROM EMPLOYEES AS E
    JOIN DEPARTMENTS AS D
        ON E.DEPARTMENT_ID = D.DEPARTMENT_ID
ORDER BY
    E.salary
;
""")

Hay un miembro menos. El inner join no ha encontrado match

Vamos a verificar cuántos empleados tienen un `DEPARTMENT_ID` nulo, lo cual explicaría por qué no fueron incluidos en el `INNER JOIN`.

In [ ]:
SQL("""
SELECT
    COUNT(*) AS Employees_Without_Department
FROM
    EMPLOYEES
WHERE
    DEPARTMENT_ID IS NULL;
""")

Efectivamente, la ausencia de este empleado indica un Empleado sin departamento, omitido por el inner join.

Un método sin usar el comando `INNER JOIN` sería usando `FROM-WHERE`

In [ ]:
SQL("""
SELECT
    EMPLOYEES.first_name,
    EMPLOYEES.Department_ID,
    DEPARTMENTS.Department_ID,
    DEPARTMENTS.Department_Name
FROM
    EMPLOYEES, DEPARTMENTS
WHERE
    EMPLOYEES.Department_ID = DEPARTMENTS.Department_ID
    ;
""")

In [ ]:
SQL("""
SELECT
    EMPLOYEES.first_name AS Nombre,
    EMPLOYEES.last_name AS Apellidos,
    EMPLOYEES.email AS Email,
    EMPLOYEES.salary AS Salario,
    DEPARTMENTS.Department_Name
FROM
    EMPLOYEES, DEPARTMENTS
WHERE
    EMPLOYEES.Department_ID = DEPARTMENTS.Department_ID
    ;
""")

## Exploración Propia: Contar Empleados por Departamento

Quiero probar a obtener el nombre del departamento y el número de empleados que pertenecen a cada uno. Sería conveniente para entender el funcionamiento de los join.

In [ ]:
SQL("""
SELECT
    Dep.department_name AS NombreDepartamento,
    COUNT(Emp.employee_id) AS NumeroDeEmpleados
FROM
    DEPARTMENTS AS Dep
INNER JOIN
    EMPLOYEES AS Emp
    ON
        Dep.department_id = Emp.department_id
GROUP BY
    Dep.department_name
ORDER BY
    NumeroDeEmpleados DESC
;
""")
# Algorithm actions

## Ejercicio f)
Obtener una lista de aquellos apellidos de empleados que no comiencen por la letra e ni terminen por la letra n, pero que contengan la letra t ordenada alfabéticamente por apellido de la A a la Z.

In [ ]:
SQL("""
SELECT DISTINCT
    last_name AS Apellido
FROM
    EMPLOYEES
WHERE
    last_name NOT LIKE 'E%'
    AND last_name NOT LIKE '%n'
    AND last_name LIKE '%t%'
ORDER BY
    last_name ASC;
""")

## Ejercicio g)
Mostrar Nombre y Apellidos en una única columna con el formato 'Apellidos, Nombre' para todos los empleados cuyo teléfono contenga el prefijo 515.

In [ ]:
SQL("""
SELECT
    last_name || ', ' || first_name AS "Nombre Completo"
FROM
    EMPLOYEES
WHERE
    phone_number LIKE '515%'
;
""")

# NOTA: EN ACCESS concatenar Sería
# (E.last_name & ', ' & E.first_name) AS NombreCompleto


## Ejercicio h)
Obtener una lista de aquellos apellidos de empleados que están repetidos y no terminen por la letra h o r, indicando el número de repeticiones.

In [ ]:
SQL("""
SELECT
    last_name AS Apellidos,
    COUNT(last_name) AS Repeticiones
FROM EMPLOYEES
WHERE
    last_name NOT LIKE '%h'
    AND
    last_name NOT LIKE '%r'
GROUP BY
    last_name
HAVING
    Repeticiones > 1
;
""")

## Ejercicio i)
Obtener una lista con los apellidos, email y nombre del departamento de los empleados directores de departamento ordenada por el nombre del departamento del cual sean director.

$$
A := ｛ Apellidos, Email, Departamento｝
$$
$$
E := 𝛒(EMPLOYEES)
$$
$$
D := 𝛒(DEPARTMENTS)
$$
$$
𝛼 := (E.EMPLOYEE＿ID = D.MANAGER＿ID)
$$
$$
ED :=
  E
  \underset{𝛼 }{⋈}
  D
$$
$$
R₁ :=
𝜋(A)[ED]
$$
$$
𝜏(DEPARTMENT_NAME↑)[R₁]
$$

In [ ]:
SQL("""
SELECT
    E.LAST_NAME AS Apellidos,
    E.EMAIL AS Email,
    D.DEPARTMENT_NAME AS Departamento
FROM EMPLOYEES AS E,
    DEPARTMENTS AS D
WHERE
    E.EMPLOYEE_ID = D.MANAGER_ID
ORDER BY
    D.DEPARTMENT_NAME
;
""")

## Ejercicio j)
Obtener una lista con los Nombres y Apellidos de los directores de departamento que superen el salario medio de todos los directores de departamento de la empresa, indicando también el nombre de la región dónde el departamento está ubicado.

In [ ]:
SQL("""
SELECT
    E.FIRST_NAME AS Nombre,
    E.LAST_NAME AS Apellidos,
    D.DEPARTMENT_NAME AS Departamento,
    R.REGION_NAME AS Region
FROM EMPLOYEES AS E,
    DEPARTMENTS AS D,
    LOCATIONS AS L,
    COUNTRIES AS C,
    REGIONS AS R
WHERE
    E.EMPLOYEE_ID = D.MANAGER_ID
    AND
    D.LOCATION_ID = L.LOCATION_ID
    AND
    L.COUNTRY_ID = C.COUNTRY_ID
    AND
    C.REGION_ID = R.REGION_ID
    AND
    E.SALARY > (SELECT AVG(E2.SALARY)
        FROM EMPLOYEES AS E2, DEPARTMENTS AS D2
        WHERE E2.EMPLOYEE_ID = D2.MANAGER_ID)
ORDER BY
    D.DEPARTMENT_NAME
;
""")

## Ejercicio k)
Obtener una lista con los nombres de los departamentos con las Ciudades, Nombre de País y Nombre de la Región a donde pertenezcan ordenadas alfabéticamente por el nombre del país de la Z a la A.

In [ ]:
SQL("""
SELECT
    D.Department_Name AS Departamento,
    L.City AS Ciudad,
    C.Country_Name AS Pais,
    R.Region_Name AS Region
FROM
    DEPARTMENTS AS D,
    LOCATIONS AS L,
    COUNTRIES AS C,
    REGIONS AS R
WHERE
    D.Location_ID = L.Location_ID
    AND
    L.Country_ID = C.Country_ID
    AND
    C.Region_ID = R.Region_ID

ORDER BY C.COUNTRY_NAME DESC
;
""")

## Ejercicio l)
Obtener una lista con los nombres y apellidos de los empleados, así como el nombre de su categoría laboral (JOB_TITLE) y el salario mínimo de ésta, ordenado de menor a mayor salario mínimo.

1. EJ := E⋈J (E.Job_ID=J.Job_ID)
2. 𝜋(Nombres, Apellidos, Puesto, Salario_Minimo)[EJ]
3. 𝜏 [2.] (Salario_Minimo↑)

In [ ]:
SQL("""
SELECT
    E.First_Name AS Nombre,
    E.Last_Name AS Apellidos,
    J.Job_Title AS Puesto,
    J.Min_Salary AS "Salario Mínimo"
FROM
    EMPLOYEES AS E,
    JOBS AS J
WHERE E.JOB_ID = J.JOB_ID
ORDER BY J.MIN_SALARY ASC
;
""")

## Ejercicio m)
Obtener una lista de categoría laborales indicando: el nombre, el número de empleados que pertenecen, el mayor salario pagado, el menor salario pagado y el salario medio todo ello referente a la mencionada categoría laboral ordenada de mayor a menor salario medio.

1. E := 𝜌(EMPLOYEES)
2. J := 𝜌(JOBS)
3. EJ := E⋈J(E.JOB_ID=J.JOB_ID)
1. Categorías := 𝛾{J.Job_Title,|E.Job_ID|, J.Max_Salario, J.Min_Salario, AVG(E.Salario)[EJ]}
2. R₁ := 𝜋(Puesto, N_Empleados,  Max_Salario, Min_Salario, Med_Salario)[Categorías]
3. R := 𝜏 (Med_Salario↓)[R₁]


In [ ]:
SQL("""
SELECT
    J.Job_Title AS Puesto,
    COUNT(E.Job_ID) as "Número de Empleados",
    J.MAX_SALARY AS "Salario Máximo",
    J.MIN_SALARY AS "Salario Mínimo",
    AVG(E.Salary) as "Salario Medio"
FROM
    JOBS AS J,
    EMPLOYEES AS E
WHERE
    J.JOB_ID = E.JOB_ID
GROUP BY
    J.JOB_ID
ORDER BY
    "Salario Medio" DESC
;""")

Me preguntaba también si sería posible hacerlo con los datos reales de max/min en la tabla:

In [ ]:
SQL("""
SELECT
    J.JOB_TITLE AS Puesto,
    COUNT(E.JOB_ID) AS NumeroEmpleados,
    MAX(E.SALARY) AS SalarioMaximo,
    MIN(E.SALARY) AS SalarioMinimo,
    AVG(E.SALARY) AS SalarioMedio
FROM
    JOBS AS J,
    EMPLOYEES AS E
WHERE
    J.JOB_ID = E.JOB_ID
GROUP BY
    J.JOB_ID, J.JOB_TITLE
ORDER BY
    SalarioMedio DESC;
""")

He visto que funciona igual en `GROUP BY` si se selecciona cualquiera de los `J.JOB_ID, J.JOB_TITLE` o ambos. Me ha parecido interesante porque ambos pueden identificar el Trabajo, pero se podría granular varios modelos de Job_Title similares, o bajo el mismo nombre, pero con descriptivos diferentes. Esto puede ser util si se tiene un grupo de trabajo comercial orientado al público y otro orientado a empresas, por ejemplo.

## Ejercicio n)
Listar los nombres de todos los departamentos y el nombre de su director, incluyendo los departamentos que no tienen director asignado, en este último caso la columna dirá ‘Sin Director Asignado’.

* D := 𝜌(DEPARTMENTS)
* E := 𝜌(EMPLOYEES)
* 𝛼 := D.MANAGER_ID = E.EMPLOYEE_ID
* R₁ := D ⋈ E (𝛼)
* P₁ := 𝜋(
    - D.DEPARTMENT_NAME,
    - E.FIRST_NAME,
    - E.LAST_NAME)
    - [R₁]
* S := 𝜎(D.MANAGER_ID = ␀)[P₁]
* P₂ := 𝜋(
    - D.DEPARTMENT_NAME,
    - "Sin Director Asignado",
    - "-")
    - [S]
* R := P₁ ∪ P₂

  

In [ ]:
SQL("""
SELECT DISTINCT
    D.DEPARTMENT_NAME AS Departamento,
    E.FIRST_NAME AS "Nombre Director",
    E.LAST_NAME AS "Apellidos Director"
FROM DEPARTMENTS AS D,
    EMPLOYEES AS E
WHERE
    D.MANAGER_ID = E.EMPLOYEE_ID
UNION
SELECT
    D2.DEPARTMENT_NAME AS Departamento,
    "Sin Director Asignado" AS "Nombre Director",
    "-" AS "Apellidos Director"
FROM DEPARTMENTS AS D2
WHERE D2.MANAGER_ID IS NULL
ORDER BY "Departamento"
;

""")

## Ejercicio o)
Obtener una lista con los nombres y apellidos de los empleados, así como el nombre y apellidos de su supervisor (mánager) ordenada alfabéticamente por el código de la categoría laboral (JOB_ID) de este último.

In [ ]:
SQL("""
SELECT
    E.First_Name AS Nombre,
    E.Last_Name AS Apellidos,
    M.First_Name AS "Nombre Supervisor",
    M.Last_Name AS "Apellidos Supervisor"
FROM
    EMPLOYEES as E,
    EMPLOYEES as M
WHERE
    E.MANAGER_ID = M.EMPLOYEE_ID
ORDER BY
    M.JOB_ID
;
""")

## Ejercicio p)
Obtener una lista con los nombres de los departamentos y la ciudad donde éste está ubicado incluyendo el importe total de salarios que ese departamento paga a sus empleados.

-

In [ ]:
SQL("""
SELECT
    D.DEPARTMENT_NAME AS Departamento,
    L.CITY AS Ciudad,
    SUM(E.SALARY) AS "Salario Total"
FROM
    DEPARTMENTS AS D,
    LOCATIONS AS L,
    EMPLOYEES AS E
WHERE
    D.LOCATION_ID = L.LOCATION_ID
    AND
    D.DEPARTMENT_ID = E.DEPARTMENT_ID
GROUP BY
    D.DEPARTMENT_ID
;
""")

## Ejercicio q)
Seleccionar los empleados que no sean supervisores de nadie dando sus Nombre y Apellidos separados por coma, así como el nombre la categoría laboral (JOB_TITLE) a la que pertenecen.

In [ ]:
# Prueba de la Intermedia Para los Managers
SQL("""
SELECT
        DISTINCT E2.MANAGER_ID
        FROM
            EMPLOYEES AS E2
        WHERE
            E2.MANAGER_ID IS NOT NULL
""")

In [ ]:
SQL("""
SELECT
    E.Last_Name || ", " || E.First_Name As NombreCompleto,
    J.Job_Title As Puesto
FROM
    EMPLOYEES AS E,
    JOBS AS J
WHERE
    E.JOB_ID = J.JOB_ID
    AND E.EMPLOYEE_ID NOT IN (SELECT
        DISTINCT E2.MANAGER_ID
        FROM
            EMPLOYEES AS E2
        WHERE
            E2.MANAGER_ID IS NOT NULL

        )
;
""")

## Ejercicio r)
Obtener una lista con los nombres de departamento, el número de empleados, el salario máximo, el mínimo, así como el promedio de cada uno de estos departamentos, ordenados según el salario medio de menor a mayor.

In [ ]:
SQL("""
SELECT
    D.DEPARTMENT_NAME AS Departamento,
    COUNT(E.EMPLOYEE_ID) AS "Número de Empleados",
    MAX(E.SALARY) AS "Salario Máximo",
    MIN(E.SALARY) AS "Salario Mínimo",
    ROUND(AVG(E.SALARY),2) AS "Salario Medio"
FROM
    DEPARTMENTS AS D,
    EMPLOYEES AS E
    WHERE
        D.DEPARTMENT_ID = E.DEPARTMENT_ID
    GROUP BY
        D.DEPARTMENT_ID
""")

## Ejercicio s)
Obtener el número de empleados contratados entre los años 1995 y 1999 que estén asignados a algún departamento.

In [ ]:
SQL("""
SELECT
    HIRE_DATE AS Fecha
FROM EMPLOYEES
WHERE HIRE_DATE LIKE("%199[5-9]%")
""")

In [ ]:
SQL("""
SELECT
    count(HIRE_DATE) AS "Número de Empleados"
FROM EMPLOYEES
WHERE
    (   HIRE_DATE LIKE("%1999%")
    OR
        HIRE_DATE LIKE("%1998%")
    OR
        HIRE_DATE LIKE("%1997%")
    OR
        HIRE_DATE LIKE("%1996%")
    OR
        HIRE_DATE LIKE("%1995%")
    ) AND
        DEPARTMENT_ID IS NOT NULL
""")

## Ejercicio t)
Obtener el número de ciudades que tiene cada país, indicando el nombre de ésta.

In [ ]:
SQL("""
SELECT
    C.COUNTRY_NAME AS Pais,
    COUNT(L.CITY) AS "Número de Ciudades"
FROM
    COUNTRIES AS C,
    LOCATIONS AS L
    WHERE
        C.COUNTRY_ID = L.COUNTRY_ID
    GROUP BY
        C.COUNTRY_NAME
    ORDER BY
        "Número de Ciudades" DESC
""")

## Ejercicio u)
Obtener una lista de Ciudades junto al Nombre de País y Nombre de la Región a donde pertenezcan, en la cual se indique el número de departamentos que ésta aloja. La lista ha de estar ordenada de menor a mayor número de departamentos.

In [ ]:
SQL("""
SELECT
    L.CITY AS Ciudad,
    C.COUNTRY_NAME AS Pais,
    R.REGION_NAME AS Region,
    COUNT(D.DEPARTMENT_ID) AS "Número de Departamentos"
FROM
    LOCATIONS AS L,
    COUNTRIES AS C,
    REGIONS AS R,
    DEPARTMENTS AS D
WHERE
    L.COUNTRY_ID = C.COUNTRY_ID
    AND
    C.REGION_ID = R.REGION_ID
    AND
    L.LOCATION_ID = D.LOCATION_ID
GROUP BY
    L.CITY
ORDER BY
    "Número de Departamentos"
;
""")

## Ejercicio v)
Obtener la lista de Ciudades en las que no se aloja ningún departamento de la empresa incluyendo el Nombre de País y Nombre de la Región.

In [ ]:
SQL("""
SELECT
    L.CITY AS Ciudad,
    C.COUNTRY_NAME AS Pais,
    R.REGION_NAME AS Region
FROM
    LOCATIONS AS L,
    COUNTRIES AS C,
    REGIONS AS R
WHERE
    L.COUNTRY_ID = C.COUNTRY_ID
    AND
    C.REGION_ID = R.REGION_ID
    AND
    L.LOCATION_ID NOT IN (SELECT
        D2.LOCATION_ID
        FROM
            DEPARTMENTS AS D2)
""")

## Ejercicio w)
Realizar una relación con los nombres de los directores de los departamentos ordenado por apellidos de la Z a la A, esta única columna se llamará Nombre Completo, apareciendo los Apellidos una coma y el Nombre.

In [ ]:
SQL("""
SELECT
    E.LAST_NAME || ', ' || E.FIRST_NAME AS "Nombre Completo",
    D.DEPARTMENT_NAME AS Departamento
FROM
    EMPLOYEES AS E,
    DEPARTMENTS AS D
WHERE
    E.EMPLOYEE_ID = D.MANAGER_ID
ORDER BY
    E.LAST_NAME DESC
;
""")

## Ejercicio x)
Obtener Apellidos, Salario y Nombre del departamento de aquellos empleados que, no siendo jefes de departamento, no superen el salario medio del departamento para el cual trabajan.

In [ ]:
SQL("""
SELECT
    E.LAST_NAME AS Apellidos,
    E.SALARY AS Salario,
    D.DEPARTMENT_NAME AS Departamento
FROM
    EMPLOYEES AS E,
    DEPARTMENTS AS D
WHERE
    E.DEPARTMENT_ID = D.DEPARTMENT_ID
    AND
    E.EMPLOYEE_ID NOT IN (SELECT
        DISTINCT E2.MANAGER_ID
        FROM
            EMPLOYEES AS E2
        WHERE
            E2.MANAGER_ID IS NOT NULL
        )
    AND
    Salario < (SELECT
        AVG(E3.SALARY)
        FROM EMPLOYEES AS E3
        WHERE E3.DEPARTMENT_ID = E.DEPARTMENT_ID
        )
;
""")

## Ejercicio y)
Realizar una lista de Nombre de Departamento y Ciudad donde éste se ubica de tal manera que ningún empleado trabaje en ellos y otra que sí tengan algún empleado.

In [ ]:
SQL("""
SELECT
    D.DEPARTMENT_NAME AS Departamento,
    L.CITY AS Ciudad,
    "Sin Empleados" AS "Empleados?"
FROM
    DEPARTMENTS AS D,
    LOCATIONS AS L
WHERE
    D.LOCATION_ID = L.LOCATION_ID
    AND
    D.DEPARTMENT_ID NOT IN (SELECT
        E.DEPARTMENT_ID
        FROM
            EMPLOYEES AS E)
UNION
SELECT
    D.DEPARTMENT_NAME AS Departamento,
    L.CITY AS Ciudad,
    "Con Empleados" AS "Empleados?"
FROM
    DEPARTMENTS AS D,
    LOCATIONS AS L
WHERE
    D.LOCATION_ID = L.LOCATION_ID
    AND
    D.DEPARTMENT_ID IN (SELECT
        E.DEPARTMENT_ID
        FROM
            EMPLOYEES AS E)
""")

## Ejercicio z)
Obtener Nombre, Apellidos y Salario de los empleados cuyo salario sea mayor que el salario medio de los compañeros que trabajan es su mismo país.

In [ ]:
SQL("""
SELECT
    E.FIRST_NAME AS Nombre,
    E.LAST_NAME AS Apellidos,
    E.SALARY AS Salario
FROM
    EMPLOYEES AS E,
    EMPLOYEES AS Empleado,
    DEPARTMENTS AS Emp_Dep,
    LOCATIONS AS Emp_Loc,
    COUNTRIES AS Emp_Countr
WHERE
    E.EMPLOYEE_ID = Empleado.EMPLOYEE_ID
    AND
    Empleado.DEPARTMENT_ID = Emp_Dep.DEPARTMENT_ID
    AND
    Emp_Dep.LOCATION_ID = Emp_Loc.LOCATION_ID
    AND
    Emp_Loc.COUNTRY_ID = Emp_Countr.COUNTRY_ID
    AND
    Empleado.SALARY > (SELECT
        AVG(Companeros.SALARY)
        FROM
            EMPLOYEES AS Companeros,
            DEPARTMENTS AS Comp_Dep,
            LOCATIONS AS Comp_Loc,
            COUNTRIES AS Comp_Countr
        WHERE
            Companeros.EMPLOYEE_ID <> Empleado.EMPLOYEE_ID
            AND
            Companeros.DEPARTMENT_ID = Comp_Dep.DEPARTMENT_ID
            AND
            Comp_Dep.LOCATION_ID = Comp_Loc.LOCATION_ID
            AND
            Comp_Loc.COUNTRY_ID = Comp_Countr.COUNTRY_ID
            AND
            Emp_Countr.COUNTRY_ID = Comp_Countr.COUNTRY_ID
        )
;
""")

## Ejercicio aa)
Obtener por cada tipo de trabajo (JOB_ID), indicando también su nombre (JOB_TITLE), el salario más bajo pagado (es decir cuál es el importe pagado a aquel empleado que menos gana).

In [ ]:
SQL("""
SELECT
        j.job_id AS "Tipo de Trabajo",
        j.job_title AS "Puesto",
        min(e.salary) AS "Menor Salario"
    From
        JOBS as j,
        EMPLOYEES as e
    where
        j.job_id = e.job_id
    group by j.job_id
    """)

## Ejercicio bb)
Listar el nombre, apellidos y salario de los empleados cuyo salario sea mayor que el salario máximo de al menos uno de los departamentos situados en su mismo país.

In [ ]:
SQL("""
SELECT
    E.first_name  AS Nombre,
    E.last_name   AS Apellidos,
    E.salary      AS Salario
FROM
    EMPLOYEES   AS E,
    DEPARTMENTS AS D,
    LOCATIONS   AS L,
    COUNTRIES   AS C
WHERE
    E.department_id   = D.department_id
    AND D.location_id = L.location_id
    AND L.country_id  = C.country_id

    AND EXISTS (
        SELECT 1
        FROM
            EMPLOYEES   AS E2,
            DEPARTMENTS AS D2,
            LOCATIONS   AS L2,
            COUNTRIES   AS C2
        WHERE
            E2.department_id  = D2.department_id
            AND D2.location_id = L2.location_id
            AND L2.country_id = C2.country_id
            AND C2.country_id = C.country_id
        GROUP BY
            D2.department_id
        HAVING
            MAX(E2.salary) < E.salary
    )
;


""")

## Ejercicio cc)
Obtener Nombre, Apellidos, Salario y Nombre del departamento, de aquellos empleados que superen el 75% del salario del jefe de su departamento.

1. Ρ:
    * $E:= ρ_{E}(EMPLOYEES)$
    * $D:= ρ_{D}(DEPARTMENTS)$
    * $S:= ρ_{S}(EMPLOYEES)$
3. $Α := (E.departmentId = D.departmentId)$
4. $Β := (D.managerId = S.employeeId)$
5. $Γ := (E.salary > 0.75 · S.salary)$
7. $(E ⨝_Α D)⨝_Β S$
   1. $R_1 := σ_{(Α ∧ Β)}[E ⨯ D ⨯ S]$
7. $R_2 := σ(Γ)[R_1]$
8. ⬦ $π(E.firstName, E.lastName, E.salary, D.departmentName)[R_2]$





In [ ]:
SQL("""
SELECT
    e.first_name AS Nombre,
    e.last_name AS Apellidos,
    e.salary AS Salario,
    d.department_name AS Departamento
FROM
    EMPLOYEES AS e,
    DEPARTMENTS AS d,
    EMPLOYEES AS j
WHERE
    e.department_id = d.department_id
    AND
    d.manager_id = j.employee_id
    AND
    e.salary > j.salary * 0.75
""")

## Ejercicio dd)
Obtener Apellidos, Salario y Nombre del departamento de aquellos empleados que trabajen en un departamento alojado en una ciudad en la cual estén alojados otros departamentos de la empresa (lo cual significa que en esa ciudad habrá al menos dos departamentos alojados) y además no superen el salario medio del departamento para el cual trabajan.

In [ ]:
SQL("""
SELECT
    E.last_name       AS Apellidos,
    E.salary          AS Salario,
    D.department_name AS Departamento
FROM
    EMPLOYEES   AS E,
    DEPARTMENTS AS D,
    LOCATIONS   AS L,
    (
        SELECT
            L2.city AS City
        FROM
            DEPARTMENTS AS D2,
            LOCATIONS   AS L2
        WHERE
            D2.location_id = L2.location_id
        GROUP BY
            L2.city
        HAVING
            COUNT(*) > 1
    ) AS Ciudades_Validas
WHERE
    E.department_id = D.department_id
    AND D.location_id = L.location_id
    AND L.city = Ciudades_Validas.City
    AND E.salary <= (
        SELECT AVG(EmpD.salary)
        FROM EMPLOYEES AS EmpD
        WHERE EmpD.department_id = E.department_id
        )
;
""")


## Ejercicio ee)
Obtener una lista de empleados incluyendo una columna con su nombre y apellidos (estos dos campos serán una cadena única estando separados por una coma) y otra con el número de compañeros que tengan el mismo trabajo que éste. Esta lista debe ordenarse alfabéticamente según el nombre de departamento al cual pertenezca, de la “Z” a la “A”. Esta lista solo afectará a empleados asignados a algún departamento.

In [ ]:
SQL("""
SELECT
    E.LAST_NAME || ', ' || E.FIRST_NAME AS "Nombre Completo",
    (SELECT
        COUNT(*)
        FROM
            EMPLOYEES AS E2
        WHERE
            E2.JOB_ID = E.JOB_ID
            )-1 AS "Número de compañeros"
FROM
    EMPLOYEES AS E,
    DEPARTMENTS AS D
WHERE
    E.DEPARTMENT_ID = D.DEPARTMENT_ID
ORDER BY
    D.DEPARTMENT_NAME DESC
;
""")

## Ejercicio ff)
Dar una lista con el nombre y apellidos de los empleados cuyo trabajo actual tenga un salario máximo menor a 12000 €, incluyendo el nombre y apellidos de su supervisor. Esta lista solo debe incluir a empleados pertenecientes a algún departamento, cuyo salario (del empleado en cuestión) sea menor que el salario promedio de todos sus compañeros de departamento con igual trabajo que éste.

In [ ]:
SQL("""
SELECT
    E.LAST_NAME AS Apellidos,
    E.FIRST_NAME AS Nombre,
    S.LAST_NAME AS Apellidos_Supervisor,
    S.FIRST_NAME AS Nombre_Supervisor
FROM
    EMPLOYEES AS E,
    EMPLOYEES AS S,
    JOBS AS J
WHERE
    E.MANAGER_ID = S.EMPLOYEE_ID
    AND
    E.JOB_ID = J.JOB_ID
    AND
    J.MAX_SALARY < 12000
    AND
    E.DEPARTMENT_ID IS NOT NULL
    AND
    E.SALARY < ( SELECT
        AVG(C.SALARY)
        FROM EMPLOYEES AS C
        WHERE
            C.DEPARTMENT_ID = E.DEPARTMENT_ID
            AND
            C.JOB_ID = E.JOB_ID
            AND
            E.EMPLOYEE_ID <> C.EMPLOYEE_ID

            )


""")

## Ejercicio gg)
Proveer de una lista de empleados que sean supervisores indicando Nombre y Apellidos de éste que incluya el número de empleados no directores de departamento que dicho supervisor supervisa, el número de empleados directores de departamento que dicho supervisor supervisa. De estas dos últimas columnas se debe excluir empleados que sean supervisores.

> Si vemos el problema por partes, se puede resolver utilizando tablas intermedias. Una de ellas debería de identificar los Subalternos de cada Supervisor. La otra, los directores Supervisados.

In [ ]:
SQL("""
    /* Subordinados NO directores */
    SELECT
        M.employee_id AS Id,
        COUNT(*)      AS NumSub,
        0             AS NumDir
    FROM
        EMPLOYEES AS M,
        EMPLOYEES AS E
    WHERE
        M.employee_id = E.manager_id
        AND E.employee_id NOT IN (
            SELECT Dep.manager_id
            FROM DEPARTMENTS AS Dep
            WHERE Dep.manager_id IS NOT NULL
        )
    GROUP BY
        M.employee_id

    UNION ALL

    /* Subordinados que SÍ son directores */
    SELECT
        M.employee_id AS Id,
        0             AS NumSub,
        COUNT(*)      AS NumDir
    FROM
        EMPLOYEES AS M,
        EMPLOYEES AS E
    WHERE
        M.employee_id = E.manager_id
        AND E.employee_id IN (
            SELECT Dep.manager_id
            FROM DEPARTMENTS AS Dep
            WHERE Dep.manager_id IS NOT NULL
        )
    GROUP BY
        M.employee_id
""")


In [ ]:
SQL("""
SELECT
    Super.last_name  AS "Apellido Supervisor",
    Super.first_name AS "Nombre Supervisor",
    SUM(Supervisados.Sub) AS "Nº Subordinados",
    SUM(Supervisados.Dir) AS "Nº Directores Supervisados"
FROM
    EMPLOYEES AS Super,
    (
        SELECT
            M.employee_id AS Id,
            COUNT(*)      AS Sub,
            0             AS Dir
        FROM EMPLOYEES AS M, EMPLOYEES AS E
        WHERE
            M.employee_id = E.manager_id
            AND E.employee_id NOT IN (
                SELECT Dep.manager_id
                FROM DEPARTMENTS AS Dep
                WHERE Dep.manager_id IS NOT NULL
            )
        GROUP BY M.employee_id

        UNION ALL

        SELECT
            M.employee_id AS Id,
            0             AS Sub,
            COUNT(*)      AS Dir
        FROM EMPLOYEES AS M, EMPLOYEES AS E
        WHERE
            M.employee_id = E.manager_id
            AND E.employee_id IN (
                SELECT Dep.manager_id
                FROM DEPARTMENTS AS Dep
                WHERE Dep.manager_id IS NOT NULL
            )
        GROUP BY M.employee_id
    ) AS Supervisados
WHERE
    Super.employee_id = Supervisados.Id
GROUP BY
    Super.employee_id, Super.last_name, Super.first_name
ORDER BY
    Super.last_name, Super.first_name
;


""")

## Ejercicio hh)
Realizar una lista de directores de departamento con sus nombres y apellidos, incluyendo: el número de empleados pertenecientes al departamento que dirige y que son supervisores, el sueldo medio de todos los empleados de su departamento que no son supervisores. Esta lista sólo debe incluir directores que superen en un 10% el salario promedio del resto de directores de departamento de la empresa que no sean supervisores.

He explorado como hacerla con tablas intermedias:

In [ ]:
# Supervisores
SQL("""
SELECT DISTINCT
    E2.MANAGER_ID AS supervisor_id
FROM EMPLOYEES AS E2
WHERE E2.MANAGER_ID IS NOT NULL

""")

In [ ]:
# Supervisores también directores
SQL("""
SELECT
    D.MANAGER_ID   AS director_id,
    COUNT(*)       AS num_supervisores
FROM DEPARTMENTS AS D,
     EMPLOYEES   AS E
WHERE
    D.DEPARTMENT_ID = E.DEPARTMENT_ID
    AND E.EMPLOYEE_ID IN (
        SELECT DISTINCT
            E2.MANAGER_ID
        FROM EMPLOYEES AS E2
        WHERE E2.MANAGER_ID IS NOT NULL
    )
GROUP BY
    D.MANAGER_ID
""")

In [ ]:
# Directores
SQL("""
SELECT
    D.MANAGER_ID   AS director_id,
    D.DEPARTMENT_ID
FROM DEPARTMENTS AS D
WHERE D.MANAGER_ID IS NOT NULL
""")

Final

In [ ]:
SQL("""
SELECT
    Dir.first_name,
    Dir.last_name,
    Sup.num_supervisores,
    AvgNS.avg_salary_nosuper
FROM
    EMPLOYEES AS Dir,

    (
        SELECT
            D.MANAGER_ID   AS director_id,
            COUNT(*)       AS num_supervisores
        FROM DEPARTMENTS AS D,
             EMPLOYEES   AS E
        WHERE
            D.DEPARTMENT_ID = E.DEPARTMENT_ID
            AND E.EMPLOYEE_ID IN (
                SELECT DISTINCT
                    E2.MANAGER_ID
                FROM EMPLOYEES AS E2
                WHERE E2.MANAGER_ID IS NOT NULL
            )
        GROUP BY
            D.MANAGER_ID
    ) AS Sup,

    (
        SELECT
            D.MANAGER_ID   AS director_id,
            AVG(E.SALARY)  AS avg_salary_nosuper
        FROM DEPARTMENTS AS D,
             EMPLOYEES   AS E
        WHERE
            D.DEPARTMENT_ID = E.DEPARTMENT_ID
            AND E.EMPLOYEE_ID NOT IN (
                SELECT DISTINCT
                    E2.MANAGER_ID
                FROM EMPLOYEES AS E2
                WHERE E2.MANAGER_ID IS NOT NULL
            )
        GROUP BY
            D.MANAGER_ID
    ) AS AvgNS

WHERE
    Dir.EMPLOYEE_ID = Sup.director_id
    AND Dir.EMPLOYEE_ID = AvgNS.director_id
    AND Dir.SALARY > 1.10 * (
        SELECT
            AVG(E.SALARY)
        FROM EMPLOYEES AS E
        WHERE
            E.EMPLOYEE_ID IN (
                SELECT
                    D.MANAGER_ID
                FROM DEPARTMENTS AS D
                WHERE D.MANAGER_ID IS NOT NULL
            )
            AND E.EMPLOYEE_ID NOT IN (
                SELECT DISTINCT
                    E2.MANAGER_ID
                FROM EMPLOYEES AS E2
                WHERE E2.MANAGER_ID IS NOT NULL
            )
    )
ORDER BY
    Dir.last_name,
    Dir.first_name
;


""")